# Notebook 03: Paralelismo, Benchmarks y Patrones Avanzados

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/03_paralelismo_benchmarks.ipynb)

**Módulo 16 — Clase 3**

Este notebook acompaña los archivos `04b_asyncio_patrones.md`, `05_paralelismo.md`, `06_librerias_y_decision.md`.

Secciones **** se trabajan durante la sesión.  
Secciones **** se completan después.

---

In [ ]:
import asyncio
import time
import threading
import os
import sys
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

print(f'Python {sys.version}')
print(f'CPU cores disponibles: {os.cpu_count()}')

## Sección 1: create_task vs gather — trabajo intermedio "gratis"

**Hipótesis:** con `create_task`, podemos ejecutar trabajo independiente mientras una tarea espera.
El trabajo intermedio debe ocurrir "gratis" — durante el wait de la tarea lanzada.

In [ ]:
async def tarea_io(nombre: str, duracion: float) -> str:
    """Simula I/O-bound: wait(τᵢ) ≠ ∅"""
    await asyncio.sleep(duracion)
    return f"{nombre} completada"

def trabajo_cpu_ligero(n: int) -> int:
    """Trabajo CPU-bound ligero: exec(τⱼ) mientras τᵢ espera"""
    return sum(range(n))

# --- Comparación: gather vs create_task ---

# gather: barrera — espera TODAS antes de continuar
print("=== gather: barrera ===")
t0 = time.perf_counter()
r1, r2 = await asyncio.gather(
    tarea_io("A", 1.0),
    tarea_io("B", 1.0),
)
t_gather = time.perf_counter() - t0
print(f"Resultados: {r1}, {r2}")
print(f"Tiempo: {t_gather:.2f}s (esperado ~1s)")
print()

# create_task: lanzar y continuar con trabajo intermedio
print("=== create_task + trabajo intermedio ===")
t0 = time.perf_counter()

# ① Lanzar τᵢ en background
tarea_a = asyncio.create_task(tarea_io("A", 1.0))

# ② Trabajo independiente DURANTE el wait de A
t_trabajo = time.perf_counter()
resultado_cpu = trabajo_cpu_ligero(5_000_000)   # ~200ms de CPU
t_trabajo = time.perf_counter() - t_trabajo
print(f"  Trabajo CPU completado en {t_trabajo:.2f}s (durante wait de A)")

# ③ Punto de dependencia: necesitamos el resultado de A
resultado_a = await tarea_a
t_create_task = time.perf_counter() - t0

print(f"  {resultado_a}")
print(f"Tiempo total: {t_create_task:.2f}s (esperado ~max(1s, tiempo_cpu))")
print()
print(f"Observa: el trabajo CPU ({t_trabajo:.2f}s) fue 'gratis' — ocurrió durante el wait de A.")
print(f"Sin create_task, hubiera sumado {1.0 + t_trabajo:.2f}s en lugar de {t_create_task:.2f}s")

## Sección 2: as_completed — procesar resultados conforme llegan

Con tareas de **duración variable**, `gather` espera al más lento antes de procesar cualquier resultado.
`as_completed` permite procesar cada resultado tan pronto como está disponible.

In [ ]:
import random
import asyncio
import time

async def buscar_fuente(nombre: str, latencia: float) -> str:
    """Simula búsqueda en una fuente externa con latencia variable"""
    await asyncio.sleep(latencia)
    return f"resultado de {nombre} (tardó {latencia:.1f}s)"

fuentes = [
    ("Wikipedia", 0.5),
    ("ArXiv",     2.1),
    ("GitHub",    0.8),
    ("PubMed",    1.5),
    ("DuckDuck",  0.3),
]


# --- Con gather: espera el más lento ---
print("=== asyncio.gather: espera al más lento ===")
t0 = time.perf_counter()
resultados = await asyncio.gather(
      *[buscar_fuente(nombre, lat) for nombre, lat in fuentes]
  )
t_gather = time.perf_counter() - t0
print(f"Todos los resultados disponibles a los {t_gather:.2f}s")
for r in resultados:
    print(f"  {r}")
print()

# --- Con as_completed: procesar conforme llegan ---
print("=== asyncio.as_completed: procesa conforme llegan ===")
t0 = time.perf_counter()
tareas = [asyncio.create_task(buscar_fuente(n, l)) for n, l in fuentes]

orden_llegada = []
for tarea_completada in asyncio.as_completed(tareas):
    resultado = await tarea_completada
    t_llegada = time.perf_counter() - t0
    print(f"  t={t_llegada:.2f}s → {resultado}")
    orden_llegada.append(resultado)

print()
print("Observa: el orden de llegada es por latencia, no por orden de creación.")
print("El resultado más rápido (DuckDuck, 0.3s) llega primero aunque fue creado al final.")



---

## Sección 3: asyncio.Queue — productor-consumidor

Implementa el patrón productor-consumidor usando `asyncio.Queue`.

In [ ]:
# TAREA 3: Implementar el patrón productor-consumidor

import asyncio
import time

N_PETICIONES = 10
RITMO_PRODUCCION = 0.5   # segundos entre peticiones
T_PROCESAMIENTO = 0.5    # segundos que tarda cada petición

async def productor(queue: asyncio.Queue, n: int, n_workers: int):
    """Genera n peticiones a ritmo fijo y luego envía un sentinel por worker."""
    for i in range(n):
        item = f"peticion-{i+1}"
        await queue.put(item)
        print(f"  [productor] encolada {item}")
        await asyncio.sleep(RITMO_PRODUCCION)

    for _ in range(n_workers):
        await queue.put(None)

async def worker(queue: asyncio.Queue, nombre: str, resultados: list):
    """Consume peticiones de la cola hasta recibir el sentinel (None)."""
    while True:
        item = await queue.get()
        if item is None:
            queue.task_done()
            print(f"    [{nombre}] termina (sentinel)")
            break

        t0 = time.perf_counter()
        await asyncio.sleep(T_PROCESAMIENTO)
        dt = time.perf_counter() - t0
        resultados.append((nombre, item, dt))
        print(f"    [{nombre}] procesó {item} en {dt:.2f}s")
        queue.task_done()

async def simular(n_workers: int) -> float:
    print(f"\n{'='*58}")
    print(f"Simulación con {n_workers} worker(s)")
    print(f"{'='*58}")

    queue = asyncio.Queue()
    resultados = []

    t0 = time.perf_counter()

    tarea_productor = asyncio.create_task(productor(queue, N_PETICIONES, n_workers))
    tareas_workers = [
        asyncio.create_task(worker(queue, f"worker-{i+1}", resultados))
        for i in range(n_workers)
    ]

    await tarea_productor
    await queue.join()                    # Espera a que todo lo encolado ya se procese
    await asyncio.gather(*tareas_workers) # Los workers salen cuando consumen su sentinel

    t_total = time.perf_counter() - t0
    print(f"\n✓ {len(resultados)} peticiones procesadas en {t_total:.2f}s")
    return t_total

async def main():
    tiempos = {}
    for n in [1, 2, 3]:
        tiempos[n] = await simular(n)

    print(f"\n{'='*58}")
    print("RESUMEN COMPARATIVO")
    print(f"{'='*58}")
    print(f"{'Workers':<10} {'Tiempo total':>14}  {'Ganancia vs 1':>14}")

    base = tiempos[1]
    for n, t in tiempos.items():
        ganancia = "—" if n == 1 else f"{base / t:.2f}x"
        print(f"{n:<10} {t:>12.2f}s  {ganancia:>14}")

    print(f"\nProducción:    {N_PETICIONES} peticiones a razón de 1 cada {RITMO_PRODUCCION:.1f}s")
    print(f"Procesamiento: {T_PROCESAMIENTO:.1f}s por petición")
    print("Idea clave: cuando el productor ya es el cuello de botella, agregar workers deja de ayudar.")

await main()

## Sección 4: fire-and-forget — excepción silenciada vs tracked

In [ ]:
# TAREA 4: Demostrar fire-and-forget con excepción no propagada al caller

import asyncio
import gc
import logging

logging.basicConfig(level=logging.ERROR)
logger = logging.getLogger(__name__)

def handle_task_result(future: asyncio.Future) -> None:
    """Se ejecuta cuando la tarea termina y registra la excepción si existe."""
    if future.cancelled():
        logger.error("La tarea fue cancelada")
    elif (exc := future.exception()) is not None:
        logger.error("Excepción capturada en tarea fire-and-forget: %s", exc, exc_info=exc)

async def tarea_que_falla(nombre: str):
    await asyncio.sleep(0.1)
    raise ValueError(f"¡Error en la tarea {nombre}!")

async def main():
    print("=== Anti-patrón: fire-and-forget sin tracking ===")
    asyncio.create_task(tarea_que_falla("A"))
    await asyncio.sleep(0.3)
    gc.collect()
    await asyncio.sleep(0)
    print("La excepción NO se propagó al caller; en notebook suele aparecer aparte como warning del event loop.\n")

    print("=== Patrón correcto: fire-and-forget con callback ===")
    t = asyncio.create_task(tarea_que_falla("B"))
    t.add_done_callback(handle_task_result)
    await asyncio.sleep(0.3)
    print("El programa siguió corriendo y la excepción quedó registrada explícitamente.")

await main()

## Sección 5: Benchmark asyncio vs threading — I/O-bound

Compara asyncio y ThreadPoolExecutor para tareas I/O-bound.
Ambos deberían funcionar — ¿cuál es más rápido y más simple?

In [ ]:
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor

def io_bound_sync(duracion: float) -> str:
    time.sleep(duracion)
    return "resultado"

async def io_bound_async(duracion: float) -> str:
    await asyncio.sleep(duracion)
    return "resultado"

N   = 20
DUR = 0.2

# ── ThreadPoolExecutor ───────────────────────────────────────────────────────
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=N) as pool:
    futuros = [pool.submit(io_bound_sync, DUR) for _ in range(N)]
    resultados_thread = [f.result() for f in futuros]
t_thread = time.perf_counter() - t0
print(f"ThreadPoolExecutor ({N} workers): {t_thread:.2f}s  (esperado ~{DUR:.1f}s)")

# ── asyncio.gather ───────────────────────────────────────────────────────────
async def medir_asyncio():
    t0 = time.perf_counter()
    resultados_async = await asyncio.gather(*[io_bound_async(DUR) for _ in range(N)])
    t_async = time.perf_counter() - t0
    print(f"asyncio.gather     ({N} tareas):  {t_async:.2f}s  (esperado ~{DUR:.1f}s)")
    print(f"Speedup de asyncio sobre secuencial ideal (~{N*DUR:.1f}s): {(N*DUR)/t_async:.2f}x")
    print(f"Speedup de threads sobre secuencial ideal (~{N*DUR:.1f}s): {(N*DUR)/t_thread:.2f}x")
    print("\nConclusión: para I/O-bound, ambos modelos solapan esperas muy bien; asyncio suele ser más simple cuando ya trabajas con APIs async.")

await medir_asyncio()

## Sección 6: threading vs multiprocessing — CPU-bound con el GIL en números

Confirma empíricamente la predicción del GIL: threading no escala para CPU-bound, multiprocessing sí.

In [ ]:
import os
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Tarea CPU-bound real (Python puro) ───────────────────────────────────────
# Usamos un loop explícito para que el GIL sí importe.
def tarea_cpu(n: int) -> int:
    total = 0
    for i in range(n):
        total += (i * i) % 97
    return total

# ── Configuración ────────────────────────────────────────────────────────────
N_TRABAJO = 3_000_000
N_TAREAS  = 8  # misma cantidad total de trabajo en todas las corridas

cpu = os.cpu_count() or 2
N_WORKERS_LIST = sorted(set([1, 2, 4, min(8, cpu)]))

# ── Baseline secuencial (mismo trabajo total) ───────────────────────────────
t0 = time.perf_counter()
for _ in range(N_TAREAS):
    tarea_cpu(N_TRABAJO)
t_seq = time.perf_counter() - t0
print(f"Secuencial ({N_TAREAS} tareas): {t_seq:.2f}s  (baseline)\n")

def benchmark_threads(n_workers: int) -> float:
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=n_workers) as pool:
        list(pool.map(tarea_cpu, [N_TRABAJO] * N_TAREAS))
    return time.perf_counter() - t0

def benchmark_processes(n_workers: int) -> float:
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        list(pool.map(tarea_cpu, [N_TRABAJO] * N_TAREAS))
    return time.perf_counter() - t0

resultados = {
    "workers": [],
    "threads": [],
    "processes": [],
    "speedup_th": [],
    "speedup_pr": [],
}

print(f"{'Workers':>8} │ {'Threads (s)':>12} │ {'Speedup':>8} │ {'Processes (s)':>14} │ {'Speedup':>8}")
print("─" * 66)

for n in N_WORKERS_LIST:
    t_th = benchmark_threads(n)
    t_pr = benchmark_processes(n)
    sp_th = t_seq / t_th
    sp_pr = t_seq / t_pr

    resultados["workers"].append(n)
    resultados["threads"].append(t_th)
    resultados["processes"].append(t_pr)
    resultados["speedup_th"].append(sp_th)
    resultados["speedup_pr"].append(sp_pr)

    print(f"{n:>8} │ {t_th:>12.2f} │ {sp_th:>7.2f}x │ {t_pr:>14.2f} │ {sp_pr:>7.2f}x")

# ── Gráfica ──────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Threading vs Multiprocessing — Tarea CPU-bound", fontsize=14, fontweight="bold")

workers = resultados["workers"]

ax1.axhline(t_seq, color="gray", linestyle="--", linewidth=1.2, label=f"Secuencial ({t_seq:.2f}s)")
ax1.plot(workers, resultados["threads"], "o-", linewidth=2, label="Threads")
ax1.plot(workers, resultados["processes"], "s-", linewidth=2, label="Processes")
ax1.set_xlabel("Número de workers")
ax1.set_ylabel("Tiempo (s)")
ax1.set_title("Tiempo de ejecución")
ax1.legend()
ax1.xaxis.set_major_locator(ticker.FixedLocator(workers))
ax1.grid(alpha=0.3)

ideal = [min(n, N_TAREAS) for n in workers]
ax2.plot(workers, ideal, "--", color="gray", linewidth=1.2, label="Speedup ideal")
ax2.plot(workers, resultados["speedup_th"], "o-", linewidth=2, label="Threads")
ax2.plot(workers, resultados["speedup_pr"], "s-", linewidth=2, label="Processes")
ax2.axhline(1, color="black", linewidth=0.8, linestyle=":")
ax2.set_xlabel("Número de workers")
ax2.set_ylabel("Speedup  (t_seq / t_workers)")
ax2.set_title("Speedup vs baseline secuencial")
ax2.legend()
ax2.xaxis.set_major_locator(ticker.FixedLocator(workers))
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("benchmark_cpu.png", dpi=150)
plt.show()

print("\nObservación: threading casi no mejora porque el GIL impide ejecutar bytecode Python en paralelo.")
print("En cambio, multiprocessing sí puede mejorar porque cada proceso tiene su propio intérprete y su propio GIL.")
print("Gráfica guardada en benchmark_cpu.png")

## Sección 7: joblib vs ProcessPoolExecutor — misma tarea

In [ ]:
try:
    from joblib import Parallel, delayed
    JOBLIB_OK = True
except ImportError:
    print("joblib no instalado — pip install joblib")
    JOBLIB_OK = False

if JOBLIB_OK:
    datos = [1_500_000] * 16   # 16 tareas CPU-bound del mismo tamaño

    # ProcessPoolExecutor
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=4) as pool:
        resultados_ppe = list(pool.map(tarea_cpu, datos))
    t_ppe = time.perf_counter() - t0
    print(f"ProcessPoolExecutor (4 workers): {t_ppe:.2f}s")

    # joblib
    t0 = time.perf_counter()
    resultados_jl = Parallel(n_jobs=4)(delayed(tarea_cpu)(n) for n in datos)
    t_jl = time.perf_counter() - t0
    print(f"joblib.Parallel (n_jobs=4):       {t_jl:.2f}s")

    assert resultados_ppe == resultados_jl, "¡Resultados diferentes!"
    print("\nResultados idénticos: ✓")
    print("¿Cuándo preferir joblib?")
    print("- Cuando quieres una interfaz muy compacta para mapear muchas tareas.")
    print("- Cuando ya trabajas con ecosistema numérico y te conviene el backend 'loky'.")
    print("- Cuando quieres paralelizar pipelines sin manejar manualmente futuros o pools.")

## Sección 8: Anti-patrón lambda (PicklingError)

In [ ]:
# TAREA 8: Reproducir el PicklingError con lambda en ProcessPoolExecutor

from functools import partial

def al_cuadrado(x: int) -> int:
    return x ** 2

def potencia(x: int, exponente: int) -> int:
    return x ** exponente

# Anti-patrón: lambda no puede serializarse de forma portable entre procesos
try:
    with ProcessPoolExecutor(max_workers=2) as pool:
        resultados = list(pool.map(lambda x: x**2, [1, 2, 3, 4]))
    print("¿Sin error? En algunos entornos puede pasar, pero no es portable.")
    print(resultados)
except Exception as e:
    print(f"Error esperado: {type(e).__name__}: {e}")

print("\nFix 1: función a nivel de módulo")
with ProcessPoolExecutor(max_workers=2) as pool:
    resultados_fix_1 = list(pool.map(al_cuadrado, [1, 2, 3, 4]))
print(resultados_fix_1)

print("\nFix 2: functools.partial sobre función serializable")
cuadrado = partial(potencia, exponente=2)
with ProcessPoolExecutor(max_workers=2) as pool:
    resultados_fix_2 = list(pool.map(cuadrado, [1, 2, 3, 4]))
print(resultados_fix_2)

assert resultados_fix_1 == resultados_fix_2 == [1, 4, 9, 16]
print("\nAmbos fixes producen el mismo resultado sin usar lambda.")

## Sección 9: Pool por petición vs pool compartido — medir diferencia

In [ ]:
# TAREA 9: Comparar overhead de crear pool por petición vs pool compartido

N_PETICIONES = 10

# Anti-patrón: crear un pool nuevo en cada petición
t0 = time.perf_counter()
for _ in range(N_PETICIONES):
    with ProcessPoolExecutor(max_workers=2) as pool:
        list(pool.map(tarea_cpu, [300_000, 300_000]))
t_pool_por_peticion = time.perf_counter() - t0
print(f"Pool por petición ({N_PETICIONES} veces): {t_pool_por_peticion:.2f}s")

# Correcto: pool compartido
t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=2) as pool_compartido:
    for _ in range(N_PETICIONES):
        list(pool_compartido.map(tarea_cpu, [300_000, 300_000]))
t_pool_compartido = time.perf_counter() - t0
print(f"Pool compartido ({N_PETICIONES} peticiones): {t_pool_compartido:.2f}s")

print(f"\nOverhead del anti-patrón: {t_pool_por_peticion / t_pool_compartido:.1f}x más lento")
print("Conclusión: crear/destruir procesos por petición mete overhead evitable; un pool compartido amortiza ese costo.")

## Sección 10 (opcional): run_in_executor — asyncio + ProcessPoolExecutor

Integra asyncio con ProcessPoolExecutor para manejar carga mixta (I/O + CPU).
Este es el patrón del chatbot v3 (M5b).

In [ ]:
# TAREA 10: Implementar M5b — asyncio + ProcessPoolExecutor
#
# Escenario: 10 peticiones llegan simultáneamente.
# Cada petición: consulta BD (I/O, 0.1s) + inferencia LLM local (CPU, 0.5s)
#
# Compara:
#   a) asyncio para I/O + inferencia bloqueante dentro del event loop
#   b) asyncio para I/O + run_in_executor(ProcessPoolExecutor) para CPU
#
# ¿Cuánto mejora la versión b sobre la a?
# ¿Cómo se relaciona con la Ley de Amdahl?

import os

N_USUARIOS = 10
T_IO = 0.1
T_CPU = 0.5
MAX_WORKERS = min(4, os.cpu_count() or 2)

def inferencia_local(historial: str) -> str:
    """CPU-bound: busy loop durante ~T_CPU segundos."""
    limite = time.perf_counter() + T_CPU
    acumulador = 0
    while time.perf_counter() < limite:
        acumulador += 1
    return f"respuesta para {historial} ({acumulador} iteraciones)"

async def consulta_bd(usuario: int) -> str:
    await asyncio.sleep(T_IO)
    return f"historial-{usuario}"

# a) Versión bloqueante: el CPU corre dentro del event loop
async def handle_request_bloqueante(usuario: int) -> str:
    historial = await consulta_bd(usuario)
    return inferencia_local(historial)

# b) Versión correcta: el CPU se manda a procesos aparte
async def handle_request_executor(usuario: int, pool: ProcessPoolExecutor) -> str:
    historial = await consulta_bd(usuario)
    loop = asyncio.get_running_loop()
    return await loop.run_in_executor(pool, inferencia_local, historial)

async def version_a():
    t0 = time.perf_counter()
    resultados = await asyncio.gather(*[handle_request_bloqueante(i) for i in range(N_USUARIOS)])
    return resultados, time.perf_counter() - t0

async def version_b():
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as pool:
        resultados = await asyncio.gather(
            *[handle_request_executor(i, pool) for i in range(N_USUARIOS)]
        )
    return resultados, time.perf_counter() - t0

async def main():
    resultados_a, t_a = await version_a()
    resultados_b, t_b = await version_b()

    speedup = t_a / t_b
    p = MAX_WORKERS

    # Amdahl: speedup = 1 / (S + (1-S)/p)
    if p > 1:
        s_implicita = (1 / speedup - 1 / p) / (1 - 1 / p)
        s_implicita = max(0.0, min(1.0, s_implicita))
    else:
        s_implicita = 1.0

    print(f"Versión a) bloqueante en event loop: {t_a:.2f}s")
    print(f"Versión b) run_in_executor + procesos ({p} workers): {t_b:.2f}s")
    print(f"Mejora observada: {speedup:.2f}x")
    print(f"Fracción secuencial implícita por Amdahl: S ≈ {s_implicita:.2%}")

    print("\nInterpretación:")
    print("- La parte I/O (0.1s) ya se solapa bien con asyncio en ambas versiones.")
    print("- Lo que destruye el rendimiento en la versión a es meter CPU bloqueante dentro del event loop.")
    print("- La versión b mejora porque paraleliza el tramo CPU en procesos; aun así no llega al ideal por overhead y por la parte no paralelizable.")

await main()